In [3]:
import torch
import triton
import triton.language as tl
import time

# -------------------------------------------------
# Triton GEMM kernel
# -------------------------------------------------
@triton.jit
def matmul_kernel(
    A, B, C,
    N,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = pid_m * BLOCK + tl.arange(0, BLOCK)
    offs_n = pid_n * BLOCK + tl.arange(0, BLOCK)
    offs_k = tl.arange(0, BLOCK)

    acc = tl.zeros((BLOCK, BLOCK), dtype=tl.float32)

    for k in range(0, N, BLOCK):
        a_ptrs = A + offs_m[:, None] * stride_am + (k + offs_k[None, :]) * stride_ak
        b_ptrs = B + (k + offs_k[:, None]) * stride_bk + offs_n[None, :] * stride_bn

        a = tl.load(a_ptrs, mask=(offs_m[:, None] < N) & (k + offs_k[None, :] < N), other=0.0)
        b = tl.load(b_ptrs, mask=(k + offs_k[:, None] < N) & (offs_n[None, :] < N), other=0.0)

        acc += tl.dot(a, b)

    c_ptrs = C + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
    tl.store(c_ptrs, acc, mask=(offs_m[:, None] < N) & (offs_n[None, :] < N))


def triton_gemm(A, B):
    N = A.shape[0]
    C = torch.empty((N, N), device=A.device, dtype=A.dtype)

    BLOCK = 16
    grid = (triton.cdiv(N, BLOCK), triton.cdiv(N, BLOCK))

    matmul_kernel[grid](
        A, B, C,
        N,
        A.stride(0), A.stride(1),
        B.stride(0), B.stride(1),
        C.stride(0), C.stride(1),
        BLOCK=BLOCK,
    )
    return C


# -------------------------------------------------
# Benchmark
# -------------------------------------------------
N = 512
device = "cuda"

print(f"Running Triton GEMM on Colab GPU for N={N}")

A = torch.ones((N, N), device=device, dtype=torch.float32)
B = torch.ones((N, N), device=device, dtype=torch.float32)

# Warm-up
triton_gemm(A, B)
torch.cuda.synchronize()

runs = 10
times = []

for _ in range(runs):
    start = time.perf_counter()
    C = triton_gemm(A, B)
    torch.cuda.synchronize()
    end = time.perf_counter()
    times.append((end - start) * 1000)

print(f"Triton GEMM avg time: {sum(times)/runs:.3f} ms")


Running Triton GEMM on Colab GPU for N=512
Triton GEMM avg time: 0.619 ms
